In [ ]:
IN_DIR, OUT_FILE = "csv", "audit_details.csv"

import pandas as pd, re, glob, os

IN  = re.compile(r"overf\w*r[td]\b", re.I)     # Overført
OUT = re.compile(r"overf\w*res|transp", re.I)  # Overføres

num = lambda s: pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce").fillna(0).astype(int)
key = lambda s: re.sub(r"<[^>]*>|\b/?b\b|\W", "", s.lower().replace("ø", "o"))

res = []
for f in sorted(glob.glob(f"{IN_DIR}/*.csv")):
    df = pd.read_csv(f, dtype=str, keep_default_na=False)
    txt = df[["eier_bruker", "brugets_navn", "gaardens_navn"]].agg(" ".join, axis=1)
    val = num(df["mark"]) * 100 + num(df["ore"])

    cur = None
    for i, (t, v, h, g) in enumerate(zip(txt, val, df["herred"].map(key), df["gaards_no"])):
        if h != cur: cur, carry, s, last, farms = h, 0, 0, None, []   # new herred in column A -> start at 0
        short = len(t.split()) <= 3
        hit = lambda kind, exp: res.append(dict(
            file=os.path.basename(f), herred=df.at[i, "herred"], row=i + 2, check=kind,
            gaards_no=f"{farms[0]}-{farms[-1]}" if farms else "",
            expected=exp / 100, printed=v / 100, difference=(v - exp) / 100))
        if short and IN.search(t):
            if last is not None and v != last: hit("Overført ≠ previous Overføres", last)
            carry, s, farms = v, 0, []
        elif short and OUT.search(t):
            if carry + s != v: hit("Overført + rows ≠ Overføres", carry + s)
            carry = last = v; s, farms = 0, []
        else:
            s += v
            if any(c.isdigit() for c in g): farms.append(g)

out = pd.DataFrame(res).sort_values("difference", key=abs, ascending=False)
out.to_csv(OUT_FILE, index=False)
out